# 🌀 DeepInterpolation denoising (2-photon) — T4CT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leonor-sinogas/T4CT-team-8/blob/main/notebooks/deepinterp_denoise.ipynb)

Self-supervised denoising of a 2-photon movie with **DeepInterpolation**
(Lecoq et al., *Nat. Methods* 2021). Each frame is predicted from its 30
neighbours before + after; independent noise can't be predicted, so it's removed.

`Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. It runs on a
bundled **sample** 2p movie by default; point it at your own TIFF in the *data* cell.

Output frames = input − 60 (30 lost at each end, no neighbours).

In [ ]:
# 1) GPU check
!nvidia-smi -L || echo 'No GPU — Runtime > Change runtime type > T4 GPU'
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 2) Install DeepInterpolation (TF/tifffile/h5py are already on Colab)
!pip install -q deepinterpolation
import deepinterpolation; print('deepinterpolation', getattr(deepinterpolation, '__version__', 'ok'))

In [ ]:
# 3) Get a pretrained 2-photon model (Allen 'Ai93': GCaMP6, 512x512, 30 Hz)
import os
if not os.path.exists('ai93_model'):
    !wget -q -O ai93.zip 'https://www.dropbox.com/sh/vwxf1uq2j60uj9o/AAC9BQI1bdfmAL3OFO0lmVb1a?dl=1'
    !unzip -o -q ai93.zip -d ai93_model && rm ai93.zip
MODEL = 'ai93_model/2019_09_11_23_32_unet_single_1024_mean_absolute_error_Ai93-0450.h5'
print('model:', MODEL, '| exists:', os.path.exists(MODEL))

In [ ]:
# 4) Choose the movie to denoise
# --- Option B (default): bundled sample 2p movie, so this runs without your data ---
if not os.path.exists('deepinterpolation'):
    !git clone -q --depth 1 https://github.com/AllenInstitute/deepinterpolation.git
TIF = 'deepinterpolation/sample_data/ophys_tiny_761605196.tif'

# --- Option A (your data): mount Drive and point at your recording ---
# from google.colab import drive; drive.mount('/content/drive')
# TIF = '/content/drive/MyDrive/T4CT/recording.tif'

import tifffile
print('movie:', TIF, '|', tifffile.TiffFile(TIF).series[0].shape)

In [ ]:
# 5) Run inference (TIFF -> denoised HDF5). end_frame=-1 does the whole movie.
from deepinterpolation.generator_collection import SingleTifGenerator
from deepinterpolation.inference_collection import core_inference

generator_param = dict(pre_post_frame=30, pre_post_omission=0, steps_per_epoch=-1,
                       train_path=TIF, batch_size=5,
                       start_frame=0, end_frame=99,   # demo subset; -1 = full movie
                       randomize=0)
inference_param = dict(model_path=MODEL, output_file='denoised.h5')
core_inference(inference_param, SingleTifGenerator(generator_param)).run()
print('done -> denoised.h5')

In [ ]:
# 6) Compare noisy vs denoised + save a viewable TIFF
import h5py, numpy as np, matplotlib.pyplot as plt
raw = tifffile.imread(TIF).astype('float32')
with h5py.File('denoised.h5', 'r') as f:
    den = np.squeeze(f['data'][()]).astype('float32')
off = 30                      # denoised[i] corresponds to raw[off + i]
i = len(den) // 2
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, img, t in zip(ax, [raw[off + i], den[i], raw[off + i] - den[i]],
                     ['noisy', 'DeepInterpolation', 'removed (noise)']):
    a.imshow(img, cmap='gray'); a.set_title(t); a.axis('off')
plt.show()
tifffile.imwrite('denoised.tif', den)
print('raw', raw.shape, '-> denoised', den.shape, '(saved denoised.tif)')

## Notes & next steps
- **Your data:** mount Drive (cell 4, Option A) and set `TIF`. For the full movie set `end_frame=-1` in cell 5 (slower). Save `denoised.tif` back to Drive and feed it into the suite2p pipeline — compare ROI count / trace SNR vs the raw movie.
- **Pretrained vs your own:** the Ai93 model was trained on Allen GCaMP6 / 512×512 / 30 Hz data. If results look off on your recording, **train your own** model self-supervised (no ground truth needed) — see `deepinterpolation/examples/example_tiny_ophys_training.py` and **docs/deepinterpolation.md**.
- **If `load_model` errors** on a TF version mismatch, try the newer 2021 models inside `ai93_model/` (e.g. `..._feat_32_power_2_depth_4_unet_True-0100-0.5733.h5`), or pin TF (see the docs).
- **CLI equivalent:** `scripts/deepinterp_denoise.py infer --tif ... --model ... --out denoised.h5` then `... to-tif --h5 denoised.h5 --out denoised.tif`.